## Experiments

In [1]:
import stable_whisper
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever, InMemoryBM25Retriever
from haystack.components.rankers import MetaFieldRanker
from haystack.components.samplers import TopPSampler
from haystack import Pipeline

/home/jobin/Projects/multimodal_rag_summarization/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def add_id_to_segments(segments: list) -> list:
    segments_with_id = []
    for idx, segment in enumerate(segments):
        segment["id"] = idx
        segments_with_id.append(segment)
    return segments_with_id

In [3]:
tiny_whisper = stable_whisper.load_model("tiny")
result = tiny_whisper.transcribe("/home/jobin/Projects/multimodal_rag_summarization/data/test2.mp3", word_timestamps=True)
result_dict = result.to_dict()

Transcribe:   0%|          | 0/2142.76 [00:00<?, ?sec/s]

Detected language: english


Transcribe: 100%|█████████▉| 2142.75/2142.76 [00:38<00:00, 56.03sec/s]


In [4]:
result_dict["segments"] = add_id_to_segments(result_dict["segments"])

In [5]:
def count_words(text: str) -> int:
    return len(text.split(" "))

def merge_segments_by_idx(segments: list[dict], start_idx: int, end_idx: int) -> dict:
    if start_idx == end_idx:
        return segments[start_idx]
    merged_segment = {}
    merged_text = ""
    merged_words = []
    no_speech_probs = []
    for idx in range(start_idx, end_idx + 1):
        merged_text += segments[idx]["text"]
        merged_words.append(segments[idx]["words"])
        no_speech_probs.append(segments[idx]["no_speech_prob"])
    merged_segment["text"] = merged_text
    merged_segment["start"] = segments[start_idx]["start"]
    merged_segment["end"] = segments[end_idx]["end"]
    merged_segment["words"] = merged_words
    merged_segment["no_speech_prob"] = no_speech_probs
    return merged_segment

def merge_segments(segments: list) -> list:
    start_idx = 0
    end_idx = 0
    word_count = 0
    merged_segments = []
    while end_idx < len(segments):
        segment = segments[end_idx]
        word_count += count_words(segment["text"])
        if segment["text"].endswith(".") or end_idx == len(segments)-1:
            merged_segment = merge_segments_by_idx(segments, start_idx, end_idx)
            merged_segments.append(merged_segment)
            word_count = 0
            start_idx = end_idx + 1
            end_idx = start_idx
        else:
            end_idx += 1
    merged_segments = add_id_to_segments(merged_segments)
    return merged_segments

In [6]:
merged_segments = merge_segments(result_dict["segments"])

In [7]:
sum([count_words(segment["text"].strip()) for segment in result_dict["segments"]])

6124

In [8]:
sum([count_words(segment["text"].strip()) for segment in merged_segments])

6124

In [9]:
print("Average segment duration:", sum([seg["end"] - seg["start"] for seg in merged_segments])/len(merged_segments))

Average segment duration: 16.656774193548372


In [10]:
def prepare_docs(segments: list) -> list[dict]:
    docs = []
    for segment in segments:
        doc = {}
        doc["content"] = segment["text"].strip()
        
        meta_dict = {}
        meta_dict["start"] = segment["start"]
        meta_dict["end"] = segment["end"]
        meta_dict["words"] = segment["words"]
        meta_dict["no_speech_prob"] = segment["no_speech_prob"]
        meta_dict["id"] = segment["id"]
        doc["meta"] = meta_dict

        docs.append(doc)
    docs = [Document(content=doc["content"], meta=doc["meta"]) for doc in docs]
    return docs


In [11]:
merged_segments

[{'start': 0.2,
  'end': 0.72,
  'text': ' Good morning.',
  'seek': 0.0,
  'tokens': [2205, 2446, 13],
  'temperature': 0.0,
  'avg_logprob': -0.36989572507525803,
  'compression_ratio': 1.583941605839416,
  'no_speech_prob': 0.049388423562049866,
  'words': [{'word': ' Good',
    'start': 0.2,
    'end': 0.34,
    'probability': 0.04140004143118858,
    'tokens': [2205]},
   {'word': ' morning.',
    'start': 0.34,
    'end': 0.72,
    'probability': 0.916218101978302,
    'tokens': [2446, 13]}],
  'id': 0},
 {'text': ' I want your folks asked me to make one of these a day in the life videos to talk about how I do time management the day How I structure the routine to maximize productivity.',
  'start': 1.36,
  'end': 11.06,
  'words': [[{'word': ' I',
     'start': 1.36,
     'end': 1.42,
     'probability': 0.7373760938644409,
     'tokens': [286]},
    {'word': ' want',
     'start': 1.42,
     'end': 1.54,
     'probability': 0.7195875644683838,
     'tokens': [528]},
    {'word'

In [12]:
# dataset = prepare_docs(result_dict["segments"])
docs = prepare_docs(merged_segments)
document_store = InMemoryDocumentStore()

In [14]:
doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-mpnet-base-v2")
doc_embedder.warm_up()
docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])
# document_store.write_documents(docs)

Batches: 100%|██████████| 4/4 [00:00<00:00,  5.65it/s]


124

In [15]:
text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-mpnet-base-v2")

In [16]:
retriever = InMemoryEmbeddingRetriever(document_store, top_k=2)
ranker = MetaFieldRanker(meta_field="start", sort_order="ascending")

In [17]:
sampler = TopPSampler(top_p=0.95)

In [18]:
basic_pipeline = Pipeline()
# Add components to your pipeline
basic_pipeline.add_component("text_embedder", text_embedder)
basic_pipeline.add_component("retriever", retriever)
basic_pipeline.add_component("sampler", sampler)
# basic_pipeline.add_component("ranker", ranker)

# Now, connect the components to each other
basic_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_pipeline.connect("retriever.documents", "sampler.documents")
# basic_pipeline.connect("sampler.documents", "ranker.documents")


🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - sampler: TopPSampler
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> sampler.documents (List[Document])

In [19]:
def deduplicate_docs(docs: list[Document]) -> list[Document]:
    unique_ids = set()
    unique_docs = []
    for doc in docs:
        if doc.id not in unique_ids:
            unique_docs.append(doc)
            unique_ids.add(doc.id)
    return unique_docs


In [22]:
queries = ['Reflecting on the joy of creating content',
   'Appreciating the magic of technology',
   'Valuing hard work and love',
   'Expressing gratitude for sharing passion with others through podcasts',
   'Hoping message of hard work and compassion resonates with viewers']

sampled_docs = []
for query in queries:
    response = basic_pipeline.run({"text_embedder": {"text": query}})
    sampled_docs.extend(response["sampler"]["documents"])

unique_sampled_docs = deduplicate_docs(sampled_docs)

ranked_docs = ranker.run(documents=unique_sampled_docs)

for doc in ranked_docs["documents"]:
    print(doc.meta["start"], doc.meta["end"])

Batches: 100%|██████████| 1/1 [00:00<00:00, 145.49it/s]

281.08 306.02
1686.4 1701.86
1974.4 2005.12
2076.66 2113.7
2116.96 2122.04


## For Integration

In [1]:
import stable_whisper
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever, InMemoryBM25Retriever
from haystack.components.rankers import MetaFieldRanker
from haystack.components.samplers import TopPSampler
from haystack import Pipeline

/home/jobin/Projects/transcript_summarizer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
def add_id_to_segments(segments: list) -> list:
    segments_with_id = []
    for idx, segment in enumerate(segments):
        segment["id"] = idx
        segments_with_id.append(segment)
    return segments_with_id

def merge_segments_by_idx(segments: list[dict], start_idx: int, end_idx: int) -> dict:
    if start_idx == end_idx:
        return segments[start_idx]
    merged_segment = {}
    merged_text = ""
    merged_words = []
    no_speech_probs = []
    for idx in range(start_idx, end_idx + 1):
        merged_text += segments[idx]["text"]
        merged_words.append(segments[idx]["words"])
        no_speech_probs.append(segments[idx]["no_speech_prob"])
    merged_segment["text"] = merged_text
    merged_segment["start"] = segments[start_idx]["start"]
    merged_segment["end"] = segments[end_idx]["end"]
    merged_segment["words"] = merged_words
    merged_segment["no_speech_prob"] = no_speech_probs
    return merged_segment

def merge_segments(segments: list) -> list:
    start_idx = 0
    end_idx = 0
    merged_segments = []
    while end_idx < len(segments):
        segment = segments[end_idx]
        if segment["text"].endswith(".") or end_idx == len(segments)-1:
            merged_segment = merge_segments_by_idx(segments, start_idx, end_idx)
            merged_segments.append(merged_segment)
            start_idx = end_idx + 1
            end_idx = start_idx
        else:
            end_idx += 1
    merged_segments = add_id_to_segments(merged_segments)
    return merged_segments

def prepare_docs(segments: list) -> list[dict]:
    docs = []
    for segment in segments:
        doc = {}
        doc["content"] = segment["text"].strip()
        
        meta_dict = {}
        meta_dict["start"] = segment["start"]
        meta_dict["end"] = segment["end"]
        meta_dict["words"] = segment["words"]
        meta_dict["no_speech_prob"] = segment["no_speech_prob"]
        meta_dict["id"] = segment["id"]
        doc["meta"] = meta_dict

        docs.append(doc)
    docs = [Document(content=doc["content"], meta=doc["meta"]) for doc in docs]
    return docs

def load_embedder(embedder_type: str, **kwargs):
    if embedder_type == "SentenceTransformersDocumentEmbedder":
        embedder = SentenceTransformersDocumentEmbedder(**kwargs)
        embedder.warm_up()
    if embedder_type == "SentenceTransformersTextEmbedder":
        embedder = SentenceTransformersTextEmbedder(**kwargs)
    return embedder

def load_document_store(store_type: str):
    if store_type == "InMemoryDocumentStore":
        document_store = InMemoryDocumentStore()
    return document_store

def load_retriever(retriever_type: str, document_store, **kwargs):
    if retriever_type == "InMemoryEmbeddingRetriever":
        retriever = InMemoryEmbeddingRetriever(document_store, **kwargs)
    return retriever

def load_ranker(ranker_type: str, **kwargs):
    if ranker_type == "MetaFieldRanker":
        ranker = MetaFieldRanker(**kwargs)
    return ranker

def load_sampler(sampler_type: str, **kwargs):
    if sampler_type == "TopPSampler":
        sampler = TopPSampler(**kwargs)
    return sampler

def get_doc_embeddings(doc_embedder: SentenceTransformersDocumentEmbedder, docs: list[Document]) -> dict[str, list[Document]]:
    docs_with_embeddings = doc_embedder.run(docs)
    return docs_with_embeddings

def write_doc_embeddings_to_inmemory_store(docs_with_embeddings: list[Document], store: InMemoryDocumentStore):
    store.write_documents(docs_with_embeddings["documents"])

def preprocess_segments(segments: list[dict]) -> dict:
    merged_segments = merge_segments(segments)
    return merged_segments

def deduplicate_docs(docs: list[Document]) -> list[Document]:
    unique_ids = set()
    unique_docs = []
    for doc in docs:
        if doc.id not in unique_ids:
            unique_docs.append(doc)
            unique_ids.add(doc.id)
    return unique_docs

def load_pipeline(query_embedder, retriever, sampler) -> Pipeline:
    basic_pipeline = Pipeline()
    basic_pipeline.add_component("query_embedder", query_embedder)
    basic_pipeline.add_component("retriever", retriever)
    basic_pipeline.add_component("sampler", sampler)
    basic_pipeline.connect("query_embedder.embedding", "retriever.query_embedding")
    basic_pipeline.connect("retriever.documents", "sampler.documents")
    return basic_pipeline

def get_answer_docs(queries: list[str], pipeline: Pipeline) -> list[Document]:
    sampled_docs = []
    for query in queries:
        response = pipeline.run({"query_embedder": {"text": query}})
        sampled_docs.extend(response["sampler"]["documents"])

    # unique_sampled_docs = deduplicate_docs(sampled_docs)
    unique_sampled_docs = sampled_docs
    ranked_docs = ranker.run(documents=unique_sampled_docs)
    return ranked_docs["documents"]

In [3]:
tiny_whisper = stable_whisper.load_model("tiny")
result = tiny_whisper.transcribe("/home/jobin/Projects/transcript_summarizer/data/test2.mp3", word_timestamps=True)
result_dict = result.to_dict()

Transcribe:   0%|          | 0/2142.76 [00:00<?, ?sec/s]

Detected language: english


Transcribe: 100%|█████████▉| 2142.75/2142.76 [00:37<00:00, 57.59sec/s]


In [15]:
merged_segments = preprocess_segments(result_dict["segments"])

In [16]:
docs = prepare_docs(merged_segments)
doc_embedder = load_embedder(embedder_type="SentenceTransformersDocumentEmbedder", model="sentence-transformers/all-mpnet-base-v2")
query_embedder = load_embedder(embedder_type="SentenceTransformersTextEmbedder", model="sentence-transformers/all-mpnet-base-v2")
docs_with_embeddings = get_doc_embeddings(doc_embedder, docs)
document_store = load_document_store(store_type="InMemoryDocumentStore")
document_store.write_documents(docs_with_embeddings["documents"])
retriever = load_retriever(retriever_type="InMemoryEmbeddingRetriever", document_store=document_store, top_k=2)
ranker = load_ranker(ranker_type="MetaFieldRanker", meta_field="start", sort_order="ascending")
sampler = load_sampler(sampler_type="TopPSampler", top_p=0.95)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches: 100%|██████████| 4/4 [00:00<00:00,  6.52it/s]


In [29]:
basic_pipeline = load_pipeline(query_embedder, retriever, sampler)

PipelineError: Component has already been added in another Pipeline. Components can't be shared between Pipelines. Create a new instance instead.

In [30]:
queries = ['Morning mantra sets the tone for the day',
   'Four-hour focused session of deep work with minimal interruptions',
   'Standing desk and ergonomic keyboard setup',
   'Love for music, particularly guitar playing',
   'Long run at the end of the day with brown noise and audiobook',
   'Reflecting on thoughts and emotions during the run']

In [31]:
answers = get_answer_docs(queries, basic_pipeline)

Batches: 100%|██████████| 1/1 [00:00<00:00, 159.67it/s]


In [32]:
for a in answers:
    print(a.content)

I mean that really starts to give me amped up like let's let's get to work fifth part of the mantra zooming in even further I actually focus it on the day I visualize going through the rest of the day all the things I think I need to get done.
It's this weird ergonomic keyboard.
It's been guitar and in the human world.
All right got the social media and the guitar done Actually, there was a bunch of moments which just brought a smile to my face both the hilarity and the love I always really appreciate it when I check social media moderation it really does bring me joy so thank you for that Now it's time to face the demons in my mind going on a long run all the things I don't want to think about I usually start out listening to brown noise as I run it really focuses my mind Let's me think deeply and then about Two three miles into the run when I start feeling a little better.
Maybe 30 minutes depending on the day and the one hour of the running feel pretty good Not so good about 1936 19

In [27]:
answers[2].content

"I mean just even filming this silly thing it's like fun There's a piece of technology that somehow is capturing this that other people might watch and then there's like a microphone And you just the entirety of the technology everything is magical everything is magical reminding myself of that doesn't take much effort by just taking a a break taking a pause just Breathing and just saying down."